In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon, Circle
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
import ipywidgets as widgets
from IPython.display import display
import pandas as pd

plt.rcParams['figure.figsize'] = (15, 10)

def hole_positions(W, E, n):
    if n == 1:
        return np.array([W/2])
    return np.linspace(E, W-E, n)


def validate(W,H,D,T,n,d,E,G,Rt):
    if min(W,H,D,T,d,E,G,Rt) <= 0:
        raise ValueError("Усі розміри мають бути додатними.")
    if T >= min(H,D):
        raise ValueError("T має бути меншою за H і D.")
    if n < 1:
        raise ValueError("Кількість отворів має бути не меншою за 1.")
    if n > 1 and 2*E >= W:
        raise ValueError("Відступи E не залишають місця для ряду отворів.")
    if E <= d/2:
        raise ValueError("Відступ E має перевищувати радіус отвору.")
    if G <= T or G >= min(H,D)-T:
        raise ValueError("Оберіть G більше T, але менше за доступні габарити.")
    if Rt <= 0 or Rt >= W:
        raise ValueError("Перевірте товщину ребра Rt.")
    if G + d/2 >= min(H-T,D-T):
        raise ValueError("Ребро або отвори виходять за доступну область полиць.")
def add_extruded_polygon(ax, profile, z0, thickness, color="#9bbbd0", alpha=0.92):
    a = [(x,y,z0) for x,y in profile]
    b = [(x,y,z0+thickness) for x,y in profile]
    faces = [a,b]
    for i in range(len(profile)):
        j=(i+1)%len(profile)
        faces.append([a[i],a[j],b[j],b[i]])
    ax.add_collection3d(Poly3DCollection(
        faces, facecolors=color, edgecolors="#263746",
        linewidths=0.8, alpha=alpha
    ))


def add_hole_rings(ax, W, T, H, D, d, zvals, xhole, yhole):
    a=np.linspace(0,2*np.pi,100)
    r=d/2
    for z in zvals:
        # На горизонтальній полиці — коло в площині XZ
        ax.plot(xhole+r*np.cos(a), np.full_like(a,T), z+r*np.sin(a),
                color="darkred",lw=1.8)
        # На вертикальній полиці — коло в площині YZ
        ax.plot(np.full_like(a,T), yhole+r*np.cos(a), z+r*np.sin(a),
                color="darkred",lw=1.8)


def draw_dimension(ax, p1, p2, offset, label, text_offset=0):
    # Розмірна лінія для 2D-креслення; p1,p2 — точки у координатах осей
    x1,y1=p1; x2,y2=p2
    ox,oy=offset
    ax.annotate("", xy=(x2+ox,y2+oy), xytext=(x1+ox,y1+oy),
                arrowprops=dict(arrowstyle="<->",lw=1,color="black"))
    ax.text((x1+x2)/2+ox, (y1+y2)/2+oy+text_offset, label,
            ha="center", va="bottom", fontsize=9)
def build_model(W=120,H=90,D=80,T=6,n=3,d=10,E=18,G=28,Rt=5):
    validate(W,H,D,T,n,d,E,G,Rt)
    zvals=hole_positions(W,E,n)
    xhole=D*0.70
    yhole=H*0.68
    profile=[(0,0),(D,0),(D,T),(T,T),(T,H),(0,H)]

    fig=plt.figure(figsize=(16,11))
    ax=fig.add_subplot(221,projection="3d")
    top=fig.add_subplot(222)
    front=fig.add_subplot(223)
    side=fig.add_subplot(224)

    # Основний корпус
    add_extruded_polygon(ax,profile,0,W)

    # Два трикутні ребра жорсткості: переріз у XY, екструзія вздовж Z
    gusset=[(T,T),(T+G,T),(T,T+G)]
    z_g1=max(0, E-Rt/2)
    z_g2=min(W-Rt, W-E-Rt/2)
    add_extruded_polygon(ax,gusset,z_g1,Rt,color="#d5a45b",alpha=0.98)
    if abs(z_g2-z_g1)>1e-8:
        add_extruded_polygon(ax,gusset,z_g2,Rt,color="#d5a45b",alpha=0.98)

    # Отвори — умовні контури
    add_hole_rings(ax,W,T,H,D,d,zvals,xhole,yhole)

    ax.set(xlim=(0,D*1.08),ylim=(0,H*1.08),zlim=(0,W),
           xlabel="X — глибина",ylabel="Y — висота",zlabel="Z — ширина",
           title="3D-модель: корпус, ребра та отвори")
    ax.set_box_aspect((D,H,W))
    ax.view_init(elev=24,azim=-55)

    # Вид зверху: контур горизонтальної полиці; ребра показані умовно проєкцією
    top.fill([0,D,D,0],[0,0,W,W],color="#a9c4d6",alpha=.75)
    for z in zvals:
        top.add_patch(Circle((xhole,z),d/2,fill=False,color="darkred",lw=1.8))
    top.set(xlim=(-12,D+12),ylim=(-12,W+12),xlabel="X — глибина",
            ylabel="Z — ширина",title="Вигляд зверху")
    top.set_aspect("equal")
    top.grid(True)

    # Вид спереду: вертикальна полиця
    front.fill([0,W,W,0],[0,0,H,H],color="#a9c4d6",alpha=.75)
    for z in zvals:
        front.add_patch(Circle((z,yhole),d/2,fill=False,color="darkred",lw=1.8))
    front.set(xlim=(-12,W+12),ylim=(-12,H+12),xlabel="Z — ширина",
              ylabel="Y — висота",title="Вигляд спереду")
    front.set_aspect("equal")
    front.grid(True)

    # Вид збоку: L-профіль і трикутні ребра як проєкція
    closed=profile+[profile[0]]
    side.fill([p[0] for p in closed],[p[1] for p in closed],
              color="#a9c4d6",alpha=.75)
    side.plot([p[0] for p in closed],[p[1] for p in closed],color="#263746")
    tri=gusset+[gusset[0]]
    side.fill([p[0] for p in tri],[p[1] for p in tri],
              color="#d5a45b",alpha=.95)
    side.set(xlim=(-12,D+12),ylim=(-12,H+12),xlabel="X — глибина",
             ylabel="Y — висота",title="Вигляд збоку")
    side.set_aspect("equal")
    side.grid(True)

    fig.suptitle(f"L-кронштейн | W={W}; H={H}; D={D}; T={T}; "
                 f"n={n}; d={d}; E={E}; G={G}; Rt={Rt} мм",fontsize=13)
    plt.tight_layout()
    plt.show()

build_model()

Ww=widgets.IntSlider(value=120,min=80,max=180,step=5,description="W, мм")
Hh=widgets.IntSlider(value=90,min=60,max=150,step=5,description="H, мм")
Dd=widgets.IntSlider(value=80,min=60,max=140,step=5,description="D, мм")
Tt=widgets.IntSlider(value=6,min=3,max=12,step=1,description="T, мм")
nn=widgets.IntSlider(value=3,min=1,max=6,step=1,description="К-сть n")
hd=widgets.IntSlider(value=10,min=6,max=16,step=1,description="d, мм")
Ee=widgets.IntSlider(value=18,min=12,max=30,step=1,description="E, мм")
Gg=widgets.IntSlider(value=28,min=15,max=45,step=1,description="G, мм")
Rr=widgets.IntSlider(value=5,min=3,max=10,step=1,description="Rt, мм")

controls=widgets.VBox([
    widgets.HBox([Ww,Hh,Dd]),
    widgets.HBox([Tt,nn,hd]),
    widgets.HBox([Ee,Gg,Rr])
])
out=widgets.interactive_output(build_model,{
    "W":Ww,"H":Hh,"D":Dd,"T":Tt,"n":nn,"d":hd,"E":Ee,"G":Gg,"Rt":Rr
})
display(controls,out)

def dimensioned_sketch(W=120,H=90,D=80,T=6,n=3,d=10,E=18):
    zvals=hole_positions(W,E,n)
    fig,axs=plt.subplots(1,2,figsize=(14,5))

    # Фронтальний вигляд
    ax=axs[0]
    ax.add_patch(plt.Rectangle((0,0),W,H,fill=False,lw=2))
    for z in zvals:
        ax.add_patch(Circle((z,H*0.68),d/2,fill=False,color="darkred",lw=1.5))
    ax.set_xlim(-18,W+18); ax.set_ylim(-20,H+22)
    ax.set_aspect("equal"); ax.grid(True)
    ax.set_title("Фронтальний ескіз")
    ax.set_xlabel("Ширина, мм"); ax.set_ylabel("Висота, мм")
    draw_dimension(ax,(0,0),(W,0),(0,-12),f"W = {W}")
    draw_dimension(ax,(0,0),(0,H),(-12,0),f"H = {H}",text_offset=0)

    # Бічний вигляд
    ax=axs[1]
    prof=[(0,0),(D,0),(D,T),(T,T),(T,H),(0,H),(0,0)]
    ax.fill([p[0] for p in prof],[p[1] for p in prof],
            facecolor="#a9c4d6",alpha=.5)
    ax.plot([p[0] for p in prof],[p[1] for p in prof],lw=2,color="#263746")
    ax.add_patch(Polygon([(T,T),(T+min(28,D-T-2),T),
                         (T,T+min(28,H-T-2))],
                        closed=True,facecolor="#d5a45b",alpha=.8))
    ax.set_xlim(-18,D+20); ax.set_ylim(-20,H+22)
    ax.set_aspect("equal"); ax.grid(True)
    ax.set_title("Бічний профіль")
    ax.set_xlabel("Глибина, мм"); ax.set_ylabel("Висота, мм")
    draw_dimension(ax,(0,0),(D,0),(0,-12),f"D = {D}")
    draw_dimension(ax,(0,0),(T,0),(0,-5),f"T = {T}",text_offset=-3)

    fig.suptitle(f"Параметричний ескіз | отвори: n={n}, d={d}, крайній відступ E={E} мм")
    plt.tight_layout()
    plt.show()

dimensioned_sketch()

def calculate_geometry(W,H,D,T,n,d,E,G,Rt):
    if min(W,H,D,T,d,E,G,Rt) <= 0:
        raise ValueError("Усі параметри мають бути додатними.")
    if T >= min(H,D):
        raise ValueError("Товщина T має бути меншою за H і D.")
    if n < 1:
        raise ValueError("Кількість отворів n має бути не меншою за 1.")
    if E <= d/2 or (n > 1 and 2*E >= W):
        raise ValueError("Перевірте відступ E та діаметр отвору.")
    if G <= T or G >= min(H,D)-T:
        raise ValueError("Некоректний катет ребра G.")
    if Rt >= W:
        raise ValueError("Товщина ребра Rt має бути меншою за W.")

    r=d/2
    # Перевірка розташування отворів на полицях
    xhole=0.70*D
    yhole=0.68*H
    if xhole-r <= T or xhole+r >= D:
        raise ValueError("Отвір горизонтальної полиці наближається до межі/згину.")
    if yhole-r <= T or yhole+r >= H:
        raise ValueError("Отвір вертикальної полиці наближається до межі/згину.")

    zvals=np.array([W/2]) if n==1 else np.linspace(E,W-E,n)
    if n>1 and np.min(np.diff(zvals)) < d:
        raise ValueError("Отвори в ряду перетинаються: збільште W або зменште n чи d.")

    A_L=T*(D+H-T)
    V_body=A_L*W
    V_one_gusset=0.5*G*G*Rt
    V_gussets=2*V_one_gusset
    V_one_hole=np.pi*r*r*T
    V_holes=2*n*V_one_hole
    V_total=V_body+V_gussets-V_holes

    if V_total <= 0:
        raise ValueError("Розрахунковий об'єм має бути додатним.")

    return {
        "area_L_mm2":A_L,
        "body_mm3":V_body,
        "one_gusset_mm3":V_one_gusset,
        "gussets_mm3":V_gussets,
        "one_hole_mm3":V_one_hole,
        "holes_mm3":V_holes,
        "total_mm3":V_total,
        "total_cm3":V_total/1000,
        "hole_centers_z":zvals
    }

base = calculate_geometry(120,90,80,6,3,10,18,28,5)
base

MATERIALS = {
    "Конструкційна сталь": 7.85,
    "Нержавіюча сталь": 7.90,
    "Алюмінієвий сплав": 2.70
}

def mass_table(geometry):
    rows=[]
    for name,rho in MATERIALS.items():
        rows.append({
            "Матеріал":name,
            "Густина, г/см³":rho,
            "Маса, г":geometry["total_cm3"]*rho,
            "Маса, кг":geometry["total_cm3"]*rho/1000
        })
    return pd.DataFrame(rows)

mass_table(base)

def show_volume_breakdown(g):
    labels=["Основний корпус","Два ребра жорсткості","Матеріал отворів"]
    vals=[g["body_mm3"],g["gussets_mm3"],g["holes_mm3"]]
    fig,ax=plt.subplots(figsize=(9,5))
    bars=ax.bar(labels, np.array(vals)/1000)
    ax.set_ylabel("Об'єм, см³")
    ax.set_title("Складові розрахунку об'єму")
    ax.grid(axis="y",alpha=.3)
    for b,v in zip(bars,vals):
        ax.text(b.get_x()+b.get_width()/2,b.get_height(),
                f"{v/1000:.2f}",ha="center",va="bottom")
    plt.xticks(rotation=10)
    plt.tight_layout()
    plt.show()

show_volume_breakdown(base)
def interactive_calculation(W=120,H=90,D=80,T=6,n=3,d=10,E=18,G=28,Rt=5):
    try:
        g=calculate_geometry(W,H,D,T,n,d,E,G,Rt)
        print(f"Площа L-профілю: {g['area_L_mm2']:.1f} мм²")
        print(f"Об'єм корпусу: {g['body_mm3']/1000:.2f} см³")
        print(f"Об'єм двох ребер: {g['gussets_mm3']/1000:.2f} см³")
        print(f"Об'єм, вилучений отворами: {g['holes_mm3']/1000:.2f} см³")
        print(f"ПІДСУМКОВИЙ ОБ'ЄМ: {g['total_cm3']:.2f} см³")
        display(mass_table(g).round(4))
        show_volume_breakdown(g)
    except ValueError as e:
        print("Помилка параметрів:",e)

wW=widgets.IntSlider(value=120,min=80,max=180,step=5,description="W")
wH=widgets.IntSlider(value=90,min=60,max=150,step=5,description="H")
wD=widgets.IntSlider(value=80,min=60,max=140,step=5,description="D")
wT=widgets.IntSlider(value=6,min=3,max=12,step=1,description="T")
wn=widgets.IntSlider(value=3,min=1,max=6,step=1,description="n")
wd=widgets.IntSlider(value=10,min=6,max=16,step=1,description="d")
wE=widgets.IntSlider(value=18,min=12,max=30,step=1,description="E")
wG=widgets.IntSlider(value=28,min=15,max=45,step=1,description="G")
wRt=widgets.IntSlider(value=5,min=3,max=10,step=1,description="Rt")

controls=widgets.VBox([
    widgets.HBox([wW,wH,wD]),
    widgets.HBox([wT,wn,wd]),
    widgets.HBox([wE,wG,wRt])
])
out=widgets.interactive_output(interactive_calculation,{
    "W":wW,"H":wH,"D":wD,"T":wT,"n":wn,"d":wd,"E":wE,"G":wG,"Rt":wRt
})
display(controls,out)
def compare_designs(W=120,H=90,D=80,T=6,n=3,d=10,E=18,G=28,Rt=5,
                    material="Конструкційна сталь"):
    g=calculate_geometry(W,H,D,T,n,d,E,G,Rt)
    rho=MATERIALS[material]
    V_plain=g["body_mm3"]
    V_with_gussets=V_plain+g["gussets_mm3"]
    V_final=g["total_mm3"]
    labels=["Корпус без ребер/отворів",
            "Корпус + ребра",
            "Корпус + ребра − отвори"]
    volumes=[V_plain,V_with_gussets,V_final]
    masses=[v/1000*rho for v in volumes]
    df=pd.DataFrame({
        "Варіант":labels,
        "Об'єм, см³":np.array(volumes)/1000,
        "Маса, г":masses
    })
    display(df.round(3))

    added_mass=g["gussets_mm3"]/1000*rho
    removed_mass=g["holes_mm3"]/1000*rho
    print(f"Маса, додана двома ребрами: {added_mass:.2f} г")
    print(f"Маса, вилучена отворами: {removed_mass:.2f} г")
    print(f"Чиста зміна маси відносно корпусу без ребер та отворів: "
          f"{(g['total_mm3']-g['body_mm3'])/1000*rho:+.2f} г")

compare_designs()
